# PCS Workshop Intro — Build a Neural Network That Reads Handwriting  ✍️🔢

You will build **one Python file** that learns to recognise handwritten digits.
You write every line that matters — the layers, the loss, the update. PyTorch
only keeps score.

> **The one idea:** forward pass → loss → backpropagation → update weights → repeat

## How this notebook works

| | |
|---|---|
| **STEP** cells | Where you type. Each one matches a numbered slide. Type the code from the slide, then run the cell (**Shift+Enter**). |
| **Checkpoint** cells | Prove the stage works. They rebuild your file and test it in a fresh Python process. |
| **Catch up** cells | At the very bottom. Only if the facilitator says so. |

There are **5 STEP cells** and **3 checkpoints**. Every STEP cell prints a
progress checklist, so you can always see where you are.

You never need the file panel — everything happens in the cells below, in order.

The imports, the random seed, and a small `report()` printing helper are
written for you by the setup cell. Everything else you type.

## Setup  ·  run this once

Leave the runtime on the standard **CPU** setting — no GPU needed.

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

import torch
import torchvision

REPO_URL = "https://github.com/uh-pcs/Workshop3.git"
REPO_DIR = Path("/content/Workshop3")
PARTS_DIR = Path("/content/parts")
WORK_FILE = Path("/content/mnist_network.py")

STEPS = {
    "01_linear.py":  "STEP 1   Linear         (slide 5)",
    "02_relu.py":    "STEP 2   ReLU           (slide 6)",
    "03_network.py": "STEP 3   NeuralNetwork  (slide 7)",
    "04_data.py":    "STEP 4   load_data      (slide 9)",
    "05_main.py":    "STEP 5   main           (slide 12)",
}

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

sys.path.insert(0, str(REPO_DIR / "facilitator"))
from build_file import assemble

PARTS_DIR.mkdir(exist_ok=True)
# The imports and the random seed are boilerplate - you get those for free.
shutil.copy(REPO_DIR / "facilitator/parts/00_header.py", PARTS_DIR / "00_header.py")
shutil.copy(REPO_DIR / "solution/inspect_model.py", "/content/inspect_model.py")


def progress():
    """Print which STEP cells have been run, and what is still to come."""
    print("your file so far")
    print("-" * 40)
    for name, label in STEPS.items():
        body = (PARTS_DIR / name)
        written = body.exists() and any(
            line.strip() and not line.strip().startswith("#")
            for line in body.read_text().splitlines())
        print(f"  {'[x]' if written else '[ ]'}  {label}")


def build():
    """Rebuild mnist_network.py from every STEP you have written so far."""
    WORK_FILE.write_text(assemble(PARTS_DIR))
    return WORK_FILE


def run_check(test_code, label):
    """Rebuild the file, then test it in a brand-new Python process."""
    build()
    ok = subprocess.run([sys.executable, "-c", test_code], cwd="/content").returncode == 0
    print()
    print(f"{label} PASSED" if ok else
          f"{label} is not passing yet - read the hint above, fix the STEP cell, "
          "run it, then run this again.")
    print()
    progress()


def recover(step):
    """Catch up: copy the reference code for STEP 1..step. Facilitator only."""
    names = list(STEPS)[:step]
    if not names:
        raise ValueError("step must be 1, 2, 3, 4, or 5")
    for name in names:
        shutil.copy(REPO_DIR / "facilitator/parts" / name, PARTS_DIR / name)
    build()
    print(f"Caught up through STEP {step}.\n")
    progress()


try:
    for train in (True, False):
        torchvision.datasets.MNIST("/content/data", train=train, download=True)
    data_status = "MNIST ready"
except Exception as error:
    data_status = f"MNIST DOWNLOAD FAILED - {error!r}\n   Re-run this cell, or switch networks."

print(f"Python {sys.version.split()[0]}   PyTorch {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}  (we use CPU on purpose)")
print(data_status)
print()
print("READY - go to STEP 1 below." if data_status == "MNIST ready"
      else "SETUP NEEDS ATTENTION - see the line above.")
print()
progress()

---

# STEP 1 of 5 — `Linear`  ·  slide 5

A layer is one learned matrix multiplication plus a bias. Type the `Linear`
class from the slide into the cell below, **under the comment line**, then run
the cell.

The first line (`%%writefile …`) is not Python — it tells Colab to save this
cell into your file. Leave it exactly where it is.

In [ ]:
%%writefile /content/parts/01_linear.py
# STEP 1  ·  slide 5  ·  class Linear
# Type the class from the slide below this line, then run this cell.

### Checkpoint 1 — three images through one layer

Rebuilds your file and pushes three blank images through your layer. It should
turn `(3, 784)` into `(3, 128)`.

In [ ]:
run_check(r"""
import torch
try:
    from mnist_network import Linear
except ImportError as e:
    raise SystemExit(f"No class named Linear yet - did you run the STEP 1 cell?\n   {e}")
except Exception as e:
    raise SystemExit(f"Your file has an error:\n   {type(e).__name__}: {e}")

layer = Linear(784, 128)
out = layer.forward(torch.zeros(3, 784))
if tuple(out.shape) != (3, 128):
    raise SystemExit(f"Output shape is {tuple(out.shape)}, expected (3, 128).\n"
                     "   The weight matrix is (in_features, out_features) = (784, 128).")
if not all(p.requires_grad for p in layer.parameters()):
    raise SystemExit("A parameter is not tracked. Call .requires_grad_() on the "
                     "weights and the bias.")
print("Linear:  (3, 784) -> (3, 128),  2 tracked parameters")
""", "Checkpoint 1")

---

# STEP 2 of 5 — `ReLU`  ·  slide 6

Two lines. Without a bend, stacked linear layers collapse into one linear layer.
No checkpoint of its own — Checkpoint 2 covers it.

In [ ]:
%%writefile /content/parts/02_relu.py
# STEP 2  ·  slide 6  ·  class ReLU
# Type the class from the slide below this line, then run this cell.

---

# STEP 3 of 5 — `NeuralNetwork`  ·  slide 7

Three `Linear` layers, two `ReLU`s, and a `parameters()` method that collects
all six tensors. **Type all three methods** — `__init__`, `forward`, and
`parameters` — the checkpoint needs every one.

In [ ]:
%%writefile /content/parts/03_network.py
# STEP 3  ·  slide 7  ·  class NeuralNetwork
# Type the class from the slide below this line, then run this cell.

### Checkpoint 2 — four images become ten scores each

`(4, 784)` in, `(4, 10)` out, and `parameters()` returning six tensors.

In [ ]:
run_check(r"""
import torch
try:
    from mnist_network import NeuralNetwork
except ImportError as e:
    raise SystemExit(f"No class named NeuralNetwork yet - did you run the STEP 3 cell?\n   {e}")
except Exception as e:
    raise SystemExit(f"Your file has an error:\n   {type(e).__name__}: {e}")

model = NeuralNetwork(784, 128, 10)
out = model.forward(torch.zeros(4, 784))
if tuple(out.shape) != (4, 10):
    raise SystemExit(f"Output shape is {tuple(out.shape)}, expected (4, 10).\n"
                     "   Trace the sizes: 784 -> 128 -> 128 -> 10.")
params = model.parameters()
if len(params) != 6:
    raise SystemExit(f"parameters() returned {len(params)} tensors, expected 6 "
                     "(three weights + three biases).\n"
                     "   Add the parameters() lists together into one flat list.")
print("Network:  (4, 784) -> (4, 10),  6 trainable tensors")
""", "Checkpoint 2")

---

# STEP 4 of 5 — `load_data` and `accuracy`  ·  slide 9

Two short helpers: one loads MNIST and flattens it, the other counts how often
the biggest score names the right digit. MNIST is already downloaded, so this is
instant.

In [ ]:
%%writefile /content/parts/04_data.py
# STEP 4  ·  slide 9  ·  def load_data  and  def accuracy
# Type both functions from the slide below this line, then run this cell.

### Checkpoint 3 — your network can learn

Slides 10 and 11 explain the loss, `backward()`, and the update. This checkpoint
runs **one** of those steps on your network, on a tiny fixed problem, and checks
the loss actually went **down**.

If this passes, your gradients are wired correctly and the real training run
will work.

In [ ]:
run_check(r"""
import torch
try:
    from mnist_network import NeuralNetwork
except Exception as e:
    raise SystemExit(f"Cannot load your network:\n   {type(e).__name__}: {e}")

torch.manual_seed(0)
model = NeuralNetwork(2, 4, 2)
x = torch.tensor([[2.0, 0.0], [0.0, 2.0]])
target = torch.tensor([0, 1])

loss_before = torch.nn.functional.cross_entropy(model.forward(x), target)
loss_before.backward()

for p in model.parameters():
    if p.grad is None:
        raise SystemExit("A parameter has no gradient after backward().\n"
                         "   Check that the weights and bias call .requires_grad_() "
                         "in Linear.__init__.")
with torch.no_grad():
    for p in model.parameters():
        p -= 0.1 * p.grad
for p in model.parameters():
    p.grad.zero_()

loss_after = torch.nn.functional.cross_entropy(model.forward(x), target)
if not loss_after < loss_before:
    raise SystemExit(f"Loss went {loss_before.item():.4f} -> {loss_after.item():.4f}, "
                     "which is not down.\n   Is parameters() returning every weight "
                     "and bias?")
print(f"Learning:  loss {loss_before.item():.4f} -> {loss_after.item():.4f}")
""", "Checkpoint 3")

---

# STEP 5 of 5 — `main`  ·  slide 12

The training loop: all five moves, one visible line each. This is the last thing
you type.

Printing is handled by `report()`, which was written for you — so the loop on the
slide is nothing but the five moves.


In [ ]:
%%writefile /content/parts/05_main.py
# STEP 5  ·  slide 12  ·  def main
# Type main() from the slide below this line, then run this cell.
# Printing is already handled by report() - just call it.



## Run it — watch it learn  ·  slide 13

This rebuilds your file and runs it. Ten digits means blind guessing is right
about **10%** of the time — anything climbing well past that is your network
actually learning.

Then change `epochs = 10` to `epochs = 60` in the STEP 5 cell, run that cell
again, and run this again.

In [ ]:
build()
print(f"running {WORK_FILE} ...\n")
subprocess.run([sys.executable, str(WORK_FILE)], cwd="/content", check=True)

---

## See it read one digit  🔎

The run above happened in a separate process, so nothing survived. This trains
the same network again *here* in the notebook (about a minute) so we can look
inside it.

In [ ]:
import importlib
import inspect_model
importlib.reload(inspect_model)

model, x_test, target_test, history = inspect_model.quick_train(epochs=15)

In [ ]:
i = 0  # change this to see other test images
guess = inspect_model.predict(model, x_test[i:i + 1])[0].item()
inspect_model.show_digit(x_test[i], guess=guess, truth=target_test[i].item())

## The learning curve

Loss goes down, accuracy goes up — past the 10% blind-guess line.

In [ ]:
inspect_model.plot_history(history)

## Where it gets things wrong  ·  slide 14

Your network matches pixel patterns — it has no idea what a digit *means*. These
are the test images it got wrong while feeling **most confident**.

In [ ]:
for i in inspect_model.worst_mistakes(model, x_test, target_test, k=3):
    guess = inspect_model.predict(model, x_test[i:i + 1])[0].item()
    inspect_model.show_digit(x_test[i], guess=guess, truth=target_test[i].item())
    print("-" * 28)

---

## Done!  🎉

You built a neural network from tensors up — the learned matrix multiplication,
the activation, the loss, the gradients, and the update — and taught it to read
handwriting. No `torch.nn.Linear`, no optimizer, nothing hidden.

> It made a guess, measured how wrong it was, changed its weights a little, and
> repeated.

**Keep experimenting:** [`EXTENSIONS.md`](https://github.com/uh-pcs/Workshop3/blob/main/EXTENSIONS.md)
— change the epochs, the learning rate, the hidden size; break the
initialisation; or see the pictures the first layer learned with
`inspect_model.layer1_weight_grid(model)`.

## Keep your file  ·  slide 15

The Colab runtime is temporary, and **File → Save does not save your `.py`
file** (it would only try to write this notebook back to GitHub). Download
`mnist_network.py` now — that file is your work.

In [ ]:
from google.colab import files
files.download(str(build()))

---

## Catch up  ·  only when the facilitator says so

`recover(n)` fills in the reference code for STEP 1 through STEP n so you rejoin
the room. **It replaces your own code for those steps** — download your file
first if you want to keep it.

In [ ]:
recover(1)   # through STEP 1 - Linear

In [ ]:
recover(2)   # through STEP 2 - ReLU

In [ ]:
recover(3)   # through STEP 3 - NeuralNetwork

In [ ]:
recover(4)   # through STEP 4 - load_data

In [ ]:
recover(5)   # the complete file